In [ ]:
import json
import re
import sys
from pathlib import Path

import numpy as np
from ucimlrepo import fetch_ucirepo
from praxis import PRAXIS, ThresholdGuessBinarizer

BUILDER_DIR = Path("/home/users/zwh4/InteractiveTreeBuilder").resolve()
PUBLIC_DIR = BUILDER_DIR / "public"

sys.path.insert(0, str(BUILDER_DIR))
from python_bridge import write_praxis_builder_payload

LAMBDA_REG = 0.005
DEPTH_BUDGET = 5
RASHOMON_MULT = 0.01
MULTIPLICATIVE_SLACK = 0.0
LOOKAHEAD_K = 1
KEY_MODE = "hash"

PAYLOAD_FILENAME = "praxis_payload.js"


def accuracy(y_true, y_pred):
    return (np.asarray(y_true).astype(int) == np.asarray(y_pred).astype(int)).mean()


def force_index_html(builder_dir: Path, payload_filename: str = "praxis_payload.js"):
    index_path = builder_dir / "index.html"
    index_path.write_text(
        f"""<!doctype html>
<html lang="en">
  <head>
    <meta charset="UTF-8" />
    <meta name="viewport" content="width=device-width, initial-scale=1.0" />
    <title>PRAXIS AND/OR Builder</title>
    <script src="./{payload_filename}"></script>
  </head>
  <body>
    <div id="root"></div>
    <script type="module" src="./src/main.tsx"></script>
  </body>
</html>
""",
        encoding="utf-8",
    )


def guess_thresholds_from_feature_names(binary_feature_names):
    thresholds = {}

    for j, name in enumerate(binary_feature_names):
        s = str(name)

        found = None
        for op in ["<=", ">=", "<", ">", "="]:
            if op in s:
                rhs = s.split(op, 1)[1].strip()
                try:
                    found = float(rhs)
                except Exception:
                    found = rhs
                break

        if found is not None:
            thresholds[int(j)] = found

    return thresholds


def _extract_payload_object(js_text: str):
    m = re.search(
        r"window\.PRAXIS_BUILDER_PAYLOAD\s*=\s*(\{.*\})\s*;?\s*$",
        js_text,
        flags=re.DOTALL,
    )
    if m:
        return "payload", json.loads(m.group(1))

    gm = re.search(
        r"window\.PRAXIS_ANDOR_GRAPH\s*=\s*(\{.*?\})\s*;\s*window\.PRAXIS_ANDOR_META\s*=\s*(\{.*\})\s*;?\s*$",
        js_text,
        flags=re.DOTALL,
    )
    if gm:
        return "split_globals", {
            "graph": json.loads(gm.group(1)),
            "meta": json.loads(gm.group(2)),
        }

    raise ValueError(
        "Could not parse praxis payload JS. Expected window.PRAXIS_BUILDER_PAYLOAD = {...};"
    )


def inject_frontend_metadata(
    payload_path: Path,
    *,
    lambda_reg: float,
    depth_budget: int,
    rashomon_mult: float,
    multiplicative_slack: float,
    lookahead_k: int,
    root_n: int,
    gamma: int,
):
    js_text = payload_path.read_text(encoding="utf-8")
    _, payload = _extract_payload_object(js_text)

    graph = payload.get("graph")
    if graph is None:
        raise ValueError("Payload has no 'graph' field.")

    meta = payload.get("meta") or {}

    root_trie_id = int(graph.get("root_trie_id", 0))
    trie_nodes = graph.get("trie_nodes", [])
    root_node = None
    for node in trie_nodes:
        if int(node.get("id", -1)) == root_trie_id:
            root_node = node
            break

    root_budget_value = None
    root_min_objective = None
    root_subproblem_size = root_n

    if root_node is not None:
        root_budget_value = root_node.get("budget")
        root_min_objective = root_node.get("min_objective")
        root_subproblem_size = root_node.get("subproblem_size", root_n)

    meta.update(
        {
            "gamma": int(gamma),
            "lambda_reg": float(lambda_reg),
            "lambdaReg": float(lambda_reg),
            "root_n": int(root_n),
            "rootN": int(root_n),
            "depth_budget": int(depth_budget),
            "depthBudget": int(depth_budget),
            "rashomon_mult": float(rashomon_mult),
            "rashomonMult": float(rashomon_mult),
            "multiplicative_slack": float(multiplicative_slack),
            "multiplicativeSlack": float(multiplicative_slack),
            "lookahead_k": int(lookahead_k),
            "lookaheadK": int(lookahead_k),
        }
    )

    if root_budget_value is not None:
        meta["root_budget"] = int(root_budget_value)
        meta["rootBudget"] = int(root_budget_value)

    if root_min_objective is not None:
        meta["root_min_objective"] = int(root_min_objective)
        meta["rootMinObjective"] = int(root_min_objective)

    if root_subproblem_size is not None:
        meta["root_subproblem_size"] = int(root_subproblem_size)
        meta["rootSubproblemSize"] = int(root_subproblem_size)

    payload["meta"] = meta

    out = (
        "window.PRAXIS_BUILDER_PAYLOAD = "
        + json.dumps(payload, separators=(",", ":"))
        + ";\n"
        + "window.PRAXIS_ANDOR_GRAPH = window.PRAXIS_BUILDER_PAYLOAD.graph;\n"
        + "window.PRAXIS_ANDOR_META = window.PRAXIS_BUILDER_PAYLOAD.meta;\n"
    )

    payload_path.write_text(out, encoding="utf-8")
    return payload

magic = fetch_ucirepo(id=159)
X_raw = magic.data.features.copy()
y_raw = magic.data.targets.copy()

y_col = y_raw.columns[0]
y = (y_raw[y_col] == "g").to_numpy(np.int64)

enc = ThresholdGuessBinarizer(
    n_estimators=75,
    max_depth=2,
    random_state=0,
    column_elimination=False,
)

X = enc.fit_transform(X_raw, y).astype(np.uint8)
binning_map = enc.feature_map()

print("Original X shape: ", X_raw.shape)
print("Binarized X shape:", X.shape)
print("Binning map:", binning_map)

model = PRAXIS()

model.fit(
    X,
    y,
    lambda_reg=LAMBDA_REG,
    depth_budget=DEPTH_BUDGET,
    rashomon_mult=RASHOMON_MULT,
    multiplicative_slack=MULTIPLICATIVE_SLACK,
    key_mode=KEY_MODE,
    lookahead_k=LOOKAHEAD_K,
)

print("\nMinimum objective:", model.get_min_objective())
print("Rashomon set size:", model.count_trees())

binary_feature_names = [str(x) for x in enc.get_feature_names_out()]
raw_feature_names = [str(x) for x in enc.feature_names_in_]

continuous_groups = {
    raw_feature_names[int(raw_j)]: [int(c) for c in bin_cols]
    for raw_j, bin_cols in binning_map.items()
}

thresholds = guess_thresholds_from_feature_names(binary_feature_names)

gamma = int(round(LAMBDA_REG * X.shape[0]))

print("\nGamma / leaf penalty:", gamma)
print("Root n:", X.shape[0])


PUBLIC_DIR.mkdir(parents=True, exist_ok=True)

payload_path = write_praxis_builder_payload(
    model,
    out_dir=PUBLIC_DIR,
    feature_names=binary_feature_names,
    continuous_groups=continuous_groups,
    thresholds=thresholds,
    filename=PAYLOAD_FILENAME,
)

force_index_html(BUILDER_DIR, payload_filename=PAYLOAD_FILENAME)

payload = inject_frontend_metadata(
    Path(payload_path),
    lambda_reg=LAMBDA_REG,
    depth_budget=DEPTH_BUDGET,
    rashomon_mult=RASHOMON_MULT,
    multiplicative_slack=MULTIPLICATIVE_SLACK,
    lookahead_k=LOOKAHEAD_K,
    root_n=int(X.shape[0]),
    gamma=gamma,
)

print("\nWrote frontend payload to:")
print(payload_path)

print("\nPayload metadata now contains:")
for k in [
    "gamma",
    "lambda_reg",
    "root_n",
    "root_budget",
    "root_min_objective",
    "root_subproblem_size",
    "depth_budget",
    "rashomon_mult",
]:
    print(f"  {k}: {payload.get('meta', {}).get(k)}")

print("\nRun in terminal:")
print(f"cd {BUILDER_DIR}")
print("npm run build")
print(f"cp public/{PAYLOAD_FILENAME} dist/{PAYLOAD_FILENAME}")
print("npm run preview -- --host 0.0.0.0 --port 5173")

Original X shape:  (19020, 10)
Binarized X shape: (19020, 160)
Binning map: {0: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27], 1: [28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53], 2: [54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94], 3: [95, 96], 4: [97, 98, 99, 100, 101, 102, 103, 104, 105], 5: [106, 107], 6: [108, 109, 110, 111, 112, 113, 114, 115, 116], 7: [117, 118], 8: [119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144], 9: [145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159]}
Best objective: 3920 (0.206099)
Objective bound: 3959
Minimum objective: 3905
Cache sizes - Greedy: 774377, Lickety: 6909516, Trie: 0, Trie cache: OFF

Minimum objective: 3905
Rashomo

In [3]:
from pathlib import Path
import os, sys

print("Jupyter cwd:", Path.cwd())

BUILDER_DIR = Path("/home/users/zwh4/InteractiveTreeBuilder").resolve()
PUBLIC_DIR = BUILDER_DIR / "public"

print("Builder dir:", BUILDER_DIR)
print("Index exists:", (BUILDER_DIR / "index.html").exists())
print("Src exists:", (BUILDER_DIR / "src" / "main.tsx").exists())
print("Bridge exists:", (BUILDER_DIR / "python_bridge.py").exists())

assert (BUILDER_DIR / "index.html").exists()
assert (BUILDER_DIR / "src" / "main.tsx").exists()
assert (BUILDER_DIR / "python_bridge.py").exists()

sys.path.insert(0, str(BUILDER_DIR))
from python_bridge import write_praxis_builder_payload

Jupyter cwd: /home/users/zwh4/InteractiveTreeBuilder
Builder dir: /home/users/zwh4/InteractiveTreeBuilder
Index exists: True
Src exists: True
Bridge exists: True
